# Delta Duty 신경망 학습

`data/buck_optimal_duty_50points.csv`를 읽어 다음 매핑을 학습합니다.

$$[V_{ref}, I_{out}] \rightarrow \Delta D$$

네트워크 구조는 입력 2개, 히든레이어 2개, 각 히든레이어 8개 노드,
출력 1개입니다.

In [ ]:
from pathlib import Path
import json
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn


SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / "buck_optimal_duty_50points.csv"
MODEL_PATH = RESULTS_DIR / "duty_compensator.pt"
FIGURE_PATH = RESULTS_DIR / "duty_compensator_training.png"
METRICS_PATH = RESULTS_DIR / "duty_compensator_metrics.json"
ZIP_PATH = RESULTS_DIR / "duty_compensator_training_results.zip"

MAX_EPOCHS = 3000
LEARNING_RATE = 0.01
PATIENCE = 400

## 1. 학습 CSV 로드

CSV가 현재 런타임에 없으면 Colab 업로드 창이 열립니다. 업로드된 파일은
`data/buck_optimal_duty_50points.csv`로 저장됩니다.

In [ ]:
if not DATA_PATH.exists():
    try:
        from google.colab import files

        print(f"{DATA_PATH.name} 파일을 업로드하세요.")
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("CSV 파일이 업로드되지 않았습니다.")

        uploaded_name = next(iter(uploaded))
        DATA_PATH.write_bytes(uploaded[uploaded_name])
        print("CSV 저장 완료:", DATA_PATH)
    except ImportError as exc:
        raise FileNotFoundError(
            f"{DATA_PATH}가 없습니다. 먼저 데이터 생성 노트북을 실행하세요."
        ) from exc

df = pd.read_csv(DATA_PATH)

required_columns = {"v_ref_v", "i_out_a", "delta_duty"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"CSV 필수 열 누락: {sorted(missing_columns)}")

print(df[["v_ref_v", "i_out_a", "delta_duty"]])
print("데이터 수:", len(df))

## 2. Train/Validation 분할과 표준화

50개 데이터 중 40개를 학습에, 10개를 시험에 사용합니다. 입력과 출력은
train 데이터의 평균과 표준편차로 표준화합니다.

In [ ]:
X = df[["v_ref_v", "i_out_a"]].to_numpy(dtype=np.float32)
y = df[["delta_duty"]].to_numpy(dtype=np.float32)

rng = np.random.default_rng(SEED)
indices = rng.permutation(len(df))
train_indices = indices[:40]
val_indices = indices[40:]

X_train = X[train_indices]
y_train = y[train_indices]
X_val = X[val_indices]
y_val = y[val_indices]

x_mean = X_train.mean(axis=0)
x_std = X_train.std(axis=0)
y_mean = float(y_train.mean())
y_std = float(y_train.std())

X_train_n = (X_train - x_mean) / x_std
X_val_n = (X_val - x_mean) / x_std
y_train_n = (y_train - y_mean) / y_std
y_val_n = (y_val - y_mean) / y_std

X_train_t = torch.from_numpy(X_train_n)
y_train_t = torch.from_numpy(y_train_n)
X_val_t = torch.from_numpy(X_val_n)
y_val_t = torch.from_numpy(y_val_n)

## 3. FNN 구조: 2-8-8-1

- 히든레이어: 2개
- 히든 노드: 각 8개
- 활성화함수: ReLU
- 출력: Delta Duty 1개

In [ ]:
class DutyCompensator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 8),
            nn.ReLU(),
            nn.Linear(8, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
        )

    def forward(self, x):
        return self.net(x)


model = DutyCompensator()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-7)

parameter_count = sum(parameter.numel() for parameter in model.parameters())
mac_per_sample = 2 * 8 + 8 * 8 + 8 * 1

print(model)
print("Parameter count:", parameter_count)
print("MAC/sample:", mac_per_sample)

## 4. 모델 학습

In [ ]:
train_losses = []
val_losses = []
best_val_loss = float("inf")
best_state = None
wait = 0

for epoch in range(MAX_EPOCHS):
    model.train()
    optimizer.zero_grad()

    train_prediction = model(X_train_t)
    train_loss = criterion(train_prediction, y_train_t)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_val_t), y_val_t)

    train_losses.append(float(train_loss.item()))
    val_losses.append(float(val_loss.item()))

    if val_loss.item() < best_val_loss - 1e-10:
        best_val_loss = float(val_loss.item())
        best_state = {
            key: value.detach().clone()
            for key, value in model.state_dict().items()
        }
        wait = 0
    else:
        wait += 1

    if wait >= PATIENCE:
        break

model.load_state_dict(best_state)
model.eval()

print("학습 epoch:", len(train_losses))
print("최저 normalized validation MSE:", best_val_loss)

## 5. 성능 평가 및 `.pt` 저장

In [ ]:
X_all_n = (X - x_mean) / x_std

with torch.no_grad():
    prediction_all_n = model(torch.from_numpy(X_all_n)).numpy()

prediction_all = prediction_all_n * y_std + y_mean
error_all = prediction_all - y

train_mask = np.zeros(len(df), dtype=bool)
train_mask[train_indices] = True
val_mask = ~train_mask

metrics = {
    "train_samples": int(train_mask.sum()),
    "validation_samples": int(val_mask.sum()),
    "epochs": len(train_losses),
    "parameter_count": int(parameter_count),
    "mac_per_sample": int(mac_per_sample),
    "train_delta_duty_mae": float(np.mean(np.abs(error_all[train_mask]))),
    "validation_delta_duty_mae": float(np.mean(np.abs(error_all[val_mask]))),
    "all_delta_duty_mae": float(np.mean(np.abs(error_all))),
    "all_delta_duty_max_error": float(np.max(np.abs(error_all))),
}

checkpoint = {
    "model_state_dict": model.state_dict(),
    "architecture": [2, 8, 8, 1],
    "feature_names": ["v_ref_v", "i_out_a"],
    "target_name": "delta_duty",
    "x_mean": x_mean.tolist(),
    "x_std": x_std.tolist(),
    "y_mean": y_mean,
    "y_std": y_std,
    "metrics": metrics,
}

torch.save(checkpoint, MODEL_PATH)

with METRICS_PATH.open("w", encoding="utf-8") as file:
    json.dump(metrics, file, ensure_ascii=False, indent=2)

print(json.dumps(metrics, ensure_ascii=False, indent=2))
print("모델 저장 완료:", MODEL_PATH)

## 6. 학습 결과 Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))

axes[0].semilogy(train_losses, label="Train loss")
axes[0].semilogy(val_losses, label="Validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Normalized MSE")
axes[0].set_title("Duty compensator learning curve")
axes[0].grid(True, which="both", alpha=0.28)
axes[0].legend()

axes[1].scatter(
    y[train_mask], prediction_all[train_mask],
    s=36, alpha=0.72, label="Train"
)
axes[1].scatter(
    y[val_mask], prediction_all[val_mask],
    s=48, marker="s", alpha=0.82, label="Validation"
)

lower = min(float(y.min()), float(prediction_all.min()))
upper = max(float(y.max()), float(prediction_all.max()))
axes[1].plot([lower, upper], [lower, upper], "k--", linewidth=1.2)
axes[1].set_xlabel("Target delta duty")
axes[1].set_ylabel("Predicted delta duty")
axes[1].set_title("Delta-duty prediction")
axes[1].grid(True, alpha=0.28)
axes[1].legend()

fig.tight_layout()
fig.savefig(FIGURE_PATH, dpi=200, bbox_inches="tight")
plt.show()

print("학습 그래프 저장 완료:", FIGURE_PATH)

## 7. 모델 및 결과 다운로드

In [ ]:
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in [MODEL_PATH, FIGURE_PATH, METRICS_PATH]:
        zf.write(path, arcname=str(path))

print("학습 결과 ZIP 저장 완료:", ZIP_PATH)

try:
    from google.colab import files
    files.download(str(MODEL_PATH))
except ImportError:
    pass